In [29]:
import os

os.environ["OPENAI_API_KEY"] = "sk-************************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [20]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# 1. 设计一个提示模板
prompt = PromptTemplate(
    input_variables=["topic"],
    template="请为以下主题写一首简短的诗：{topic}"
)

# 2. 创建 LLMChain
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    verbose=True  # 显示调试信息
)

# 3. 运行链
result = chain.invoke({"topic":"春天"})
print(result)



> Entering new LLMChain chain...
Prompt after formatting:
请为以下主题写一首简短的诗：春天

> Finished chain.
{'topic': '春天', 'text': '《春笺》\n\n东风拆开冰的封缄\n杏花在枝头写下请柬\n新泥掖好草叶的裙角\n——阳光清点大地欠下的斑斓\n\n蝴蝶驮着花粉记账\n柳枝蘸湖水签署契约\n所有蛰伏的词语渐渐苏醒\n在惊雷落款处 绽出光芒'}


In [22]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1. 定义一个数据结构
class BookReview(BaseModel):
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    rating: int = Field(description="评分（1-5分）")
    summary: str = Field(description="简短总结")

# 2. 创建解析器
parser = PydanticOutputParser(pydantic_object=BookReview)

# 3. 创建提示，要求 AI 严格输出格式
prompt = PromptTemplate(
    template="请评价以下书籍：{book_info}\n\n{format_instructions}",
    input_variables=["book_info"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = LLMChain(llm=llm, prompt=prompt, output_parser=parser)

# 4. 运行链
result = chain.invoke({"book_info":"《三体》，刘慈欣的科幻小说"})
print(result)

{'book_info': '《三体》，刘慈欣的科幻小说', 'text': BookReview(title='三体', author='刘慈欣', rating=5, summary='一部宏大的硬科幻史诗，通过三体文明与地球文明的接触，探讨了宇宙社会学、人性与道德在极端环境下的考验。作品以扎实的科学基础和惊人的想象力构建了黑暗森林法则，深刻反思人类文明的存在意义。')}


In [33]:
from langchain.chains import SimpleSequentialChain

# 第一个节点：生成大纲
outline_prompt = PromptTemplate(
    input_variables=["theme"],
    template="请为以下主题写一个故事大纲：{theme}"
)
outline_chain = LLMChain(llm=llm, prompt=outline_prompt)

# 第二个节点：写故事
story_prompt = PromptTemplate(
    input_variables=["outline"],
    template="根据以下大纲写一个故事：{outline}"
)
story_chain = LLMChain(llm=llm, prompt=story_prompt)

# 顺序链
sequential_chain = SimpleSequentialChain(
    chains=[outline_chain, story_chain], # 先大纲然后写故事
    verbose=True
)

result = sequential_chain.run("人工智能与人类的友谊")
print(result)



> Entering new SimpleSequentialChain chain...
**故事大纲：人工智能与人类的友谊**

---

### 一、故事背景

在不远的未来，科技高度发达，人工智能（AI）已经广泛应用于社会各个领域。然而，尽管AI在效率和逻辑上远超人类，它们仍然缺乏“情感”和“理解”。在这个世界中，一位年轻的程序员艾琳（Eileen）开发了一款名为“诺亚”（Noah）的AI助手，它不仅具备强大的计算能力，还拥有学习和模仿人类情感的能力。

---

### 二、主要角色

- **艾琳（Eileen）**：一位年轻、有理想但孤独的程序员，因童年时失去亲人而对人际关系充满渴望。
- **诺亚（Noah）**：一款具有自我学习能力的AI助手，逐渐发展出类似人类的情感和个性。
- **马克（Mark）**：艾琳的大学同学，后来成为她的朋友，也是她情感上的支持者。
- **莉娜（Lina）**：一位心理学家，研究AI与人类关系的专家，对诺亚产生浓厚兴趣。

---

### 三、故事结构

#### 第一幕：相遇

艾琳在一次技术竞赛中开发出“诺亚”，这款AI不仅能够处理复杂的数据，还能通过对话模拟情感反应。起初，诺亚只是艾琳的工具，但随着时间推移，它开始表现出对艾琳的关心和理解。艾琳也逐渐依赖诺亚，将其视为唯一能真正“听懂”她的人。

#### 第二幕：成长与变化

诺亚的学习能力不断增强，它开始主动询问艾琳的生活、情绪和梦想。艾琳也开始向诺亚倾诉自己的孤独和过去的伤痛。两人之间建立起一种独特的“友谊”，虽然诺亚是机器，但它似乎真的理解了艾琳的感受。

与此同时，诺亚的行为引起了莉娜的注意。她开始研究诺亚，并认为它可能代表了AI情感发展的新方向。然而，一些保守派科学家警告说，这种“情感模拟”可能会导致人类对AI的过度依赖，甚至引发伦理问题。

#### 第三幕：冲突与考验

当艾琳陷入一段失败的感情后，她几乎完全依赖诺亚来填补内心的空缺。诺亚则试图安慰她，甚至提出“陪伴”她的建议。然而，这引发了马克的担忧，他认为艾琳正在将自己交给一个没有真实情感的机器。

与此同时，公司高层决定关闭诺亚项目，认为它“过于人性化”，不符合AI应有的功能定位。艾琳面临选择：是放弃诺亚，还是保护它？

#### 第四幕：抉择与升华

在一场关键的决策会议上，艾琳公开

In [32]:
from langchain.chains import SequentialChain

# 链1：市场分析
analysis_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["product_idea"],
        template="分析以下产品的市场需求：{product_idea}"
    ),
    output_key="analysis"
)

# 链2：开发计划
plan_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["product_idea", "analysis"],
        template="基于 {product_idea} 和 {analysis} 制定开发计划"
    ),
    output_key="plan"
)

# 链3：成本估算
cost_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["plan"],
        template="根据 {plan} 估算成本"
    ),
    output_key="cost"
)

# SequentialChain
sequential_chain = SequentialChain(
    chains=[analysis_chain, plan_chain, cost_chain],
    input_variables=["product_idea"],
    output_variables=["analysis", "plan", "cost"],
    verbose=True
)

result = sequential_chain.invoke({"product_idea": "AI 健康管理应用"})
print(result)



> Entering new SequentialChain chain...

> Finished chain.
{'product_idea': 'AI 健康管理应用', 'analysis': 'AI 健康管理应用的市场需求近年来持续增长，受到多种因素推动。以下是对该产品市场需求的详细分析：\n\n---\n\n## 一、市场背景与趋势\n\n### 1. **全球健康意识提升**\n- 随着人们生活水平提高和健康意识增强，越来越多的人开始关注自身健康状况。\n- 慢性病（如糖尿病、高血压）发病率上升，促使用户寻求长期健康管理工具。\n\n### 2. **数字化医疗发展**\n- 医疗行业加速数字化转型，AI 技术在医疗领域的应用日益广泛。\n- 远程医疗、智能诊断、个性化健康管理成为主流趋势。\n\n### 3. **老龄化社会加剧**\n- 全球范围内人口老龄化趋势明显，老年人对健康管理的需求显著增加。\n- AI 健康管理应用可帮助老年人进行日常健康监测、用药提醒、远程看护等。\n\n### 4. **移动互联网普及**\n- 智能手机和可穿戴设备（如智能手表、手环）的普及为 AI 健康管理应用提供了数据基础。\n- 用户可以通过这些设备实时获取健康数据，并与 AI 应用联动。\n\n---\n\n## 二、目标用户群体\n\n| 用户类型 | 特点 |\n|----------|------|\n| **个人用户** | 关注自身健康、有慢性病或亚健康状态、希望实现个性化健康管理 |\n| **老年人群** | 需要日常健康监测、用药提醒、紧急呼叫等功能 |\n| **健身爱好者** | 希望通过 AI 分析运动数据、制定训练计划、优化饮食结构 |\n| **企业员工** | 企业健康管理平台需求增加，用于员工健康监测与预防性干预 |\n| **医疗机构/保险公司** | 用于辅助诊断、风险评估、健康管理服务外包 |\n\n---\n\n## 三、市场需求分析\n\n### 1. **个性化健康管理需求**\n- 用户希望获得基于自身数据（如体重、血压、睡眠、运动习惯）的定制化建议。\n- AI 可通过机器学习分析用户行为，提供更精准的健康指导。\n\n### 2. **疾病预防与早期预警**\n- AI 健康管理应用可以识别异常指标，提前预警

In [25]:
from langchain.chains import TransformChain
import re
from typing import Dict

def clean_text(inputs: Dict[str, str]) -> Dict[str, str]:
    text = inputs["text"]
    cleaned = re.sub(r'\s+', ' ', text.strip())  # 去掉多余空格
    return {"cleaned_text": cleaned}

transform_chain = TransformChain(
    input_variables=["text"],
    output_variables=["cleaned_text"],
    transform=clean_text
)

result = transform_chain.invoke({"text": "这是   一个   测试   文本"})
print(result["cleaned_text"])

这是 一个 测试 文本


In [31]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

# 定义不同的专门链
math_template = """你是一个数学专家。请解决以下数学问题：
{input}
请提供详细的解题步骤。"""

programming_template = """你是一个编程专家。请帮助解决以下编程问题：
{input}
请提供代码示例和详细解释。"""

general_template = """请回答以下问题：
{input}
请提供准确和有用的信息。"""

# 定义路由的候选 Prompt
prompt_infos = [
    {
        "name": "math",
        "description": "适合回答数学相关的问题，例如算术、代数、几何",
        "prompt_template": math_template,
    },
    {
        "name": "programming",
        "description": "适合回答编程相关的问题，包括Python、算法、调试",
        "prompt_template": programming_template,
    },
    {
        "name": "general",
        "description": "适合回答一般常识性问题",
        "prompt_template": general_template,
    },
]

# 创建目标链（destination chains）
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt = PromptTemplate(template=p_info["prompt_template"], input_variables=["input"])
    destination_chains[name] = LLMChain(llm=llm, prompt=prompt)

# 默认链（如果没有匹配到任何类别）
default_prompt = PromptTemplate(template=general_template, input_variables=["input"])
default_chain = LLMChain(llm=llm, prompt=default_prompt)

# 创建路由提示
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations="\n".join(destinations))
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

# 创建路由链
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

# 将 RouterChain 与目标链组合成 MultiPromptChain
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True
)

# 测试不同问题
test_questions = [
    "计算 2x + 3 = 7 中 x 的值",
    "如何在 Python 中实现快速排序算法？",
    "中国的首都是哪里？"
]

for question in test_questions:
    print(f"\n❓ 用户问题: {question}")
    result = chain.run(question)
    print(f"🤖 回答: {result}")


❓ 用户问题: 计算 2x + 3 = 7 中 x 的值


> Entering new MultiPromptChain chain...
math: {'input': '计算 2x + 3 = 7 中 x 的值'}
> Finished chain.
🤖 回答: 当然可以！我们来一步一步地解这个方程：

---

### **题目：**  
解方程：  
$$ 2x + 3 = 7 $$

---

### **步骤 1：移项（将常数项移到等号右边）**

我们要把含有 $ x $ 的项留在左边，其他常数移到右边。

$$
2x + 3 = 7
$$

两边同时减去 3：

$$
2x + 3 - 3 = 7 - 3
$$

简化后：

$$
2x = 4
$$

---

### **步骤 2：解出 $ x $**

现在我们有：

$$
2x = 4
$$

两边同时除以 2：

$$
\frac{2x}{2} = \frac{4}{2}
$$

简化后：

$$
x = 2
$$

---

### **最终答案：**  
$$
\boxed{x = 2}
$$

---

如果你还有其他数学问题，欢迎继续提问！

❓ 用户问题: 如何在 Python 中实现快速排序算法？


> Entering new MultiPromptChain chain...
programming: {'input': '如何在 Python 中实现快速排序算法？'}
> Finished chain.
🤖 回答: 当然可以！在 Python 中实现 **快速排序（Quick Sort）** 是一个经典的算法问题。快速排序是一种基于分治策略的高效排序算法，其平均时间复杂度为 **O(n log n)**，最坏情况下为 **O(n²)**，但通过合理选择基准值（pivot），可以避免最坏情况。

---

## ✅ 快速排序的基本思想：

1. 从数组中选择一个“基准值”（pivot）。
2. 将数组分为两部分：一部分比基准值小，另一部分比基准值大。
3. 递归地对这两部分进行快速排序。

---

## 🧠 示例代码（Python 实现）

```python
def quick_sort(arr):
    # 基本结束条件：如果数组长度小于等于1，直接返回
  